# Recursive *vs* FFT Computation of Spline Coefficients
The tables give the number of times the recursive computation of spline coefficients is faster than computations that rely on the fast Fourier transform. The first table considers data whose lengths are powers of two. The second table considers general lengths.

In [ ]:
# Load the required libraries
from math import fsum
import numpy as np
import scipy
import time

import splinekit as sk # This library

max_degree = 9 # Maximal degree of the piecewise-polynomial splines
octaves = 14 # Number of dyadic ranges
nb_experiments = 50 # Number of experiments within a range of lengths

# Initialize the generator of random numbers
rng = np.random.default_rng()

# FFT approach
def fft_samples_to_coeff_p (
    samples,
    degree
):
    k0 = len(samples)
    if 1 == k0:
        return samples.copy()
    return scipy.linalg.solve_circulant(
        np.fromiter(
            (
                fsum(
                    sk.b_spline(k - p * k0, degree)
                    for p in range(
                        int((k - 0.5 * (degree - 1.0)) // k0),
                        int((k + 0.5 * (degree + 1.0)) // k0) + 1
                    )
                )
                for k in range(k0)
            ),
            dtype = float,
            count = k0
        ),
        samples,
        singular = "lstsq"
    )

# Table of results for daydic data lengths
print()
print("Acceleration for Dyadic Periods")
print("===========================================================")
print("   Degree |     2     3     4     5     6     7     8     9")
print("Period    |")
print("----------+------------------------------------------------")
for v in range(octaves + 1):
    # Create nb_experiments data vectors with random values and dyadic lengths
    kp = np.array([2 ** v for _ in range(nb_experiments)], dtype = int)
    data = [rng.standard_normal(k0) for k0 in kp]
    performance = [0, 0]
    for degree in range(2, max_degree + 1):
        # FFT-based approach to determine periodic spline coefficients
        fft_data = [y.copy() for y in data]
        start = time.perf_counter()
        for (experiment, samples) in enumerate(data):
            fft_data[experiment] = fft_samples_to_coeff_p(samples, degree)
        end = time.perf_counter()
        fft_duration = end - start
        # Recursive-based approach to determine periodic spline coefficients
        rec_data = [y.copy() for y in data]
        start = time.perf_counter()
        for (experiment, samples) in enumerate(data):
            c = samples.copy()
            sk.samples_to_coeff_p(c, degree = degree)
            rec_data[experiment] = c
        end = time.perf_counter()
        rec_duration = end - start
        # Relative speed
        performance += [fft_duration / rec_duration]
    print("K = {0:5d} | {2:5.1f} {3:5.1f} {4:5.1f} {5:5.1f} {6:5.1f} {7:5.1f} {8:5.1f} {9:5.1f}".format(
        2 ** v,
        2 ** v,
        performance[2],
        performance[3],
        performance[4],
        performance[5],
        performance[6],
        performance[7],
        performance[8],
        performance[9]
    ))
print("===========================================================")

# Table of results for random data lengths
print()
print()
print("Acceleration for Random Periods")
print("====================================================================")
print("            Degree |     2     3     4     5     6     7     8     9")
print("Period Range       |")
print("-------------------+------------------------------------------------")
for v in range(octaves + 1):
    # Create nb_experiments data vectors with random values and random lengths
    kp = rng.integers(low = 2 ** v, high = 2 ** (v + 1), size = nb_experiments)
    data = [rng.standard_normal(k0) for k0 in kp]
    performance = [0, 0]
    for degree in range(2, max_degree + 1):
        # FFT-based approach to determine periodic spline coefficients
        fft_data = [y.copy() for y in data]
        start = time.perf_counter()
        for (experiment, samples) in enumerate(data):
            fft_data[experiment] = fft_samples_to_coeff_p(samples, degree)
        end = time.perf_counter()
        fft_duration = end - start
        # Recursive-based approach to determine periodic spline coefficients
        rec_data = [y.copy() for y in data]
        start = time.perf_counter()
        for (experiment, samples) in enumerate(data):
            c = samples.copy()
            sk.samples_to_coeff_p(c, degree = degree)
            rec_data[experiment] = c
        end = time.perf_counter()
        rec_duration = end - start
        # Relative speed
        performance += [fft_duration / rec_duration]
    print("{0:5d} <= K < {1:5d} | {2:5.1f} {3:5.1f} {4:5.1f} {5:5.1f} {6:5.1f} {7:5.1f} {8:5.1f} {9:5.1f}".format(
        2 ** v,
        2 ** (v + 1),
        performance[2],
        performance[3],
        performance[4],
        performance[5],
        performance[6],
        performance[7],
        performance[8],
        performance[9]
    ))
print("====================================================================")


###################
#                 #
#   Be patient!   #
#                 #
###################


# Duration of the computations on a desktop computer of year 2021: ~25s

